# House Price Prediction Using Machine Learning

Course: Machine Learning for Business Applications

Program: MBA - Business Analytics

## 1. Business Problem
A real-estate business needs a data-driven way to estimate the selling price of houses based on property characteristics.

**Analytics Objective:** Develop a regression model that predicts house price.

**Target Variable:** `price`

**Business Decision:** The predicted price can support valuation, listing-price decisions and negotiations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
df = pd.read_csv("../data/Housing.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum(), df.duplicated().sum()

## 2. Feature Preparation
The target variable is `price`. Numerical predictors are area, bedrooms, bathrooms, stories and parking. Categorical predictors are mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea and furnishingstatus.

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]
num_cols = ["area", "bedrooms", "bathrooms", "stories", "parking"]
cat_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]

## 3. Exploratory Data Analysis

In [ ]:
plt.hist(df["price"], bins=25)
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.title("Distribution of House Prices")
plt.show()

In [ ]:
plt.scatter(df["area"], df["price"], alpha=0.55)
plt.xlabel("Area")
plt.ylabel("Price")
plt.title("Area vs House Price")
plt.show()

In [ ]:
df.groupby("bedrooms")["price"].mean().plot(kind="bar")
plt.xlabel("Bedrooms")
plt.ylabel("Average Price")
plt.title("Average Price by Bedrooms")
plt.show()

In [ ]:
df.groupby("airconditioning")["price"].mean().plot(kind="bar")
plt.xlabel("Air Conditioning")
plt.ylabel("Average Price")
plt.title("Average Price by Air Conditioning")
plt.show()

In [ ]:
df.groupby("furnishingstatus")["price"].mean().sort_values().plot(kind="bar")
plt.xlabel("Furnishing Status")
plt.ylabel("Average Price")
plt.title("Average Price by Furnishing Status")
plt.show()

In [ ]:
df[["price", "area", "bedrooms", "bathrooms", "stories", "parking"]].corr()["price"].sort_values(ascending=False)

## 4. Model Development

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

preprocess_lr = ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols)])
lr_model = Pipeline([("preprocess", preprocess_lr), ("model", LinearRegression())])
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

In [ ]:
preprocess_rf = ColumnTransformer([("num", "passthrough", num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
rf_model = Pipeline([("preprocess", preprocess_rf), ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

## 5. Model Evaluation

In [ ]:
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}
results = pd.DataFrame({"Linear Regression": evaluate_model(y_test, lr_pred), "Random Forest": evaluate_model(y_test, rf_pred)})
results

In [ ]:
plt.bar(["Linear Regression", "Random Forest"], [results.loc["R2", "Linear Regression"], results.loc["R2", "Random Forest"]])
plt.ylabel("R² Score")
plt.title("Model Comparison")
plt.show()

## 6. Business Interpretation
The Linear Regression model is selected because it has a higher R² score and lower error measures than Random Forest on the test data. The model can be used as a decision-support tool for estimating house prices, but the final business decision should also consider market conditions, exact location quality and property features not available in the dataset.

In [ ]:
joblib.dump(lr_model, "../model.pkl")